# CAFPO PPO Softmax Long-Only on Original Yearly Top200

This notebook is an isolated copy for testing long-only CAFPO on the original model-ready yearly top200 panel used by `cafpo_ppo_log_reproduction.ipynb`.

It does not modify or overwrite the base notebook outputs. The only CAFPO behavior change is the portfolio action mapping:

`raw actor action -> softmax(action / temperature) over valid stocks -> long-only weights -> portfolio return -> reward`

The actor may still emit positive/negative raw actions, but those are interpreted as allocation logits, not short signals. The critic and PPO reward are computed from the softmax long-only portfolio return.

In [1]:
import importlib.util
import subprocess
import sys

required = {
    "stable-baselines3[extra]": "stable_baselines3",
    "gymnasium": "gymnasium",
    "openpyxl": "openpyxl",
    "pyarrow": "pyarrow",
}
missing = [pkg for pkg, mod in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    print("Installed:", missing)
else:
    print("Notebook dependencies are already installed.")

Notebook dependencies are already installed.


In [2]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "cafpo_reproduction.py").exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "cafpo"
assert (NOTEBOOK_DIR / "cafpo_reproduction.py").exists(), NOTEBOOK_DIR
if str(NOTEBOOK_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR.resolve()))

from cafpo_reproduction import (
    OUTPUT_DIR,
    audit_model_panel,
    build_rolling_splits,
    extract_cae_outputs,
    load_model_panel,
    make_ppo_model,
    panel_to_arrays,
    performance_summary,
    run_markowitz_baseline,
    save_cae_outputs,
    set_seed,
    train_cae,
)
from cafpo_ff6_experiments import run_universe_weight_baseline
from cafpo_longonly import (
    SoftmaxLongOnlyPortfolioEnv,
    evaluate_softmax_longonly_ensemble,
    softmax_long_only_weights,
)

set_seed(42)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LONGONLY_OUTPUT_DIR = OUTPUT_DIR / "top200_softmax_longonly"
LONGONLY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LONGONLY_OUTPUT_DIR

WindowsPath('C:/xmrs_workspace/workspace/【广发】深度学习play/cafpo/rqdata_output/top200_softmax_longonly')

## 1. Load Original Model-Ready Top200 Panel

In [3]:
df, feature_cols = load_model_panel()
audit = audit_model_panel(df, feature_cols)

display(pd.Series({k: v for k, v in audit.items() if not isinstance(v, pd.Series)}))
display(audit["target_describe"])

df.head()

rows                                        46291
n_months                                      232
date_start                    2007-01-31 00:00:00
date_end                      2026-04-30 00:00:00
n_features                                     82
duplicated_date_stock_keys                      0
stocks_per_month_min                          197
stocks_per_month_max                          200
stocks_per_month_mean                  199.530172
feature_nan_count                               0
feature_inf_count                               0
feature_min                                  -1.0
feature_max                                   1.0
target_nan_count                                0
target_inf_count                                0
dtype: object

count    46291.000000
mean         0.008267
std          0.127357
min         -0.676017
1%          -0.293989
5%          -0.174842
50%          0.000000
95%          0.219705
99%          0.402200
max          3.328406
Name: ret_fwd_1m, dtype: float64

,date,next_date,year,order_book_id,top200_year_flag,mvel1,close_month_end,ret_1m,ret_fwd_1m,log_ret_fwd_1m,...,x_pchsaleinv,x_pctacc,x_ps,x_realestate,x_roavol,x_roic,x_salecash,x_stdacc,x_stdcf,x_tb
0,2007-01-31,2007-02-28,2007,000001.XSHE,1,3.722358e+10,3.447552,0.0,-0.004182,-0.004191,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2007-01-31,2007-02-28,2007,000002.XSHE,1,6.690315e+10,3.823306,0.0,-0.037231,-0.037941,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2007-01-31,2007-02-28,2007,000012.XSHE,1,1.298777e+10,3.514376,0.0,0.133698,0.125485,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2007-01-31,2007-02-28,2007,000021.XSHE,1,9.674704e+09,5.807323,0.0,0.155455,0.144494,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2007-01-31,2007-02-28,2007,000022.XSHE,1,1.225051e+10,13.223138,0.0,0.031579,0.031091,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 2. Arrays and Splits

In [4]:
arrays = panel_to_arrays(df, feature_cols, max_stocks=200)
splits = build_rolling_splits(arrays.dates, train_years=10, test_years=1)

split_table = pd.DataFrame([
    {
        "train_start_year": s.train_start_year,
        "train_end_year": s.train_end_year,
        "test_year": s.test_year,
        "n_train_months": len(s.train_idx),
        "n_test_months": len(s.test_idx),
    }
    for s in splits
])
display(split_table)

assert arrays.x.shape[1] == 200 and arrays.x.shape[2] >= 82
assert np.isfinite(arrays.x).all()
assert np.isfinite(arrays.ret_current).all()
assert np.isfinite(arrays.ret_forward).all()
assert arrays.x.min() >= -1.000001 and arrays.x.max() <= 1.000001
assert len(splits) > 0

,train_start_year,train_end_year,test_year,n_train_months,n_test_months
0,2007,2016,2017,120,12
1,2008,2017,2018,120,12
2,2009,2018,2019,120,12
3,2010,2019,2020,120,12
4,2011,2020,2021,120,12
5,2012,2021,2022,120,12
6,2013,2022,2023,120,12
7,2014,2023,2024,120,12
8,2015,2024,2025,120,12
9,2016,2025,2026,120,4


## 3. Config

In [5]:
N_FACTORS = 5
LOOKBACK = 12
CAE_EPOCHS = 200
CAE_PATIENCE = 20
PPO_TOTAL_TIMESTEPS = 20_000
PPO_N_STEPS = 64
PPO_BATCH_SIZE = 32
DRL_SEEDS = [101, 202, 303, 404, 505]
SOFTMAX_TEMPERATURE = 0.2

# Set True for full rolling CAE + PPO training.
RUN_FULL_EXPERIMENT = True
# Use [2018] or another short list for a smoke run; None means all splits.
TEST_YEARS_TO_RUN = None

## 4. Softmax Long-Only Runner

In [6]:
def make_longonly_env(factors, indices, reward_mode="log_return", temperature=SOFTMAX_TEMPERATURE):
    return SoftmaxLongOnlyPortfolioEnv(
        factors=factors,
        forward_returns=arrays.ret_forward,
        mask=arrays.mask,
        dates=arrays.dates,
        indices=indices,
        lookback=LOOKBACK,
        temperature=temperature,
        reward_mode=reward_mode,
    )


def run_top200_softmax_longonly(splits_to_run=None, total_timesteps=PPO_TOTAL_TIMESTEPS):
    if splits_to_run is None:
        splits_to_run = splits
    all_returns = []
    histories = []

    for s in splits_to_run:
        print(f"softmax long-only split test_year={s.test_year}: train {s.train_start_year}-{s.train_end_year}")
        val_idx = s.train_idx[-12:]
        fit_idx = s.train_idx[:-12]
        assert len(fit_idx) > 0 and len(val_idx) > 0
        assert arrays.dates[fit_idx].max() < arrays.dates[s.test_idx].min()

        cae_model, cae_history = train_cae(
            arrays,
            train_idx=fit_idx,
            val_idx=val_idx,
            n_factors=N_FACTORS,
            epochs=CAE_EPOCHS,
            patience=CAE_PATIENCE,
            seed=42 + int(s.test_year),
        )
        cae_history.insert(0, "test_year", s.test_year)
        histories.append(cae_history)

        factors, betas, reconstructed = extract_cae_outputs(cae_model, arrays)
        assert factors.shape == (len(arrays.dates), N_FACTORS)
        assert betas.shape[:2] == arrays.x.shape[:2]
        assert np.isfinite(factors).all() and np.isfinite(betas).all()
        save_cae_outputs(
            factors,
            betas,
            reconstructed,
            arrays,
            LONGONLY_OUTPUT_DIR / f"cafpo_softmax_longonly_cae_outputs_test_{s.test_year}",
        )

        obs, _ = make_longonly_env(factors, s.train_idx[LOOKBACK:]).reset()
        assert obs.shape == (LOOKBACK, N_FACTORS)

        models = []
        for seed in DRL_SEEDS:
            train_env = make_longonly_env(factors, s.train_idx[LOOKBACK:])
            model = make_ppo_model(
                train_env,
                seed=seed + int(s.test_year),
                n_steps=PPO_N_STEPS,
                batch_size=PPO_BATCH_SIZE,
                verbose=0,
            )
            model.learn(total_timesteps=total_timesteps, progress_bar=False)
            models.append(model)

        test_env = make_longonly_env(factors, s.test_idx)
        ppo = evaluate_softmax_longonly_ensemble(models, test_env)
        ppo.insert(0, "method", f"CAFPO_PPO_SoftmaxLongOnly_T{SOFTMAX_TEMPERATURE}_5SeedAvgAction")
        ppo.insert(1, "test_year", s.test_year)

        equal_weight = run_universe_weight_baseline(
            "EqualWeight_LongOnly",
            arrays,
            test_idx=s.test_idx,
            value_weighted=False,
        )
        value_weight = run_universe_weight_baseline(
            "ValueWeight_LongOnly",
            arrays,
            test_idx=s.test_idx,
            value_weighted=True,
        )
        markowitz_ls = run_markowitz_baseline(arrays, train_idx=s.train_idx, test_idx=s.test_idx)
        markowitz_ls["method"] = "Markowitz_LS"
        for frame in (equal_weight, value_weight, markowitz_ls):
            frame.insert(1, "test_year", s.test_year)
        all_returns.extend([equal_weight, value_weight, markowitz_ls, ppo])

    returns = pd.concat(all_returns, ignore_index=True)
    history = pd.concat(histories, ignore_index=True)
    summary = performance_summary(returns)

    returns.to_csv(LONGONLY_OUTPUT_DIR / "cafpo_ppo_softmax_longonly_top200_returns.csv", index=False, encoding="utf-8-sig")
    history.to_csv(LONGONLY_OUTPUT_DIR / "cafpo_ppo_softmax_longonly_top200_cae_history.csv", index=False, encoding="utf-8-sig")
    summary.to_csv(LONGONLY_OUTPUT_DIR / "cafpo_ppo_softmax_longonly_top200_summary.csv", index=False, encoding="utf-8-sig")
    return returns, history, summary

## 5. Run

In [7]:
if RUN_FULL_EXPERIMENT:
    years = TEST_YEARS_TO_RUN
    if years is None:
        splits_to_run = splits
    else:
        wanted = set(years)
        splits_to_run = [s for s in splits if s.test_year in wanted]
        assert splits_to_run, f"No matching splits for {years}"
    longonly_returns, longonly_cae_histories, longonly_summary = run_top200_softmax_longonly(splits_to_run)
    display(longonly_summary)
else:
    print("RUN_FULL_EXPERIMENT is False. Set it to True to train the top200 softmax long-only CAFPO run.")
    print(f"Output directory: {LONGONLY_OUTPUT_DIR}")

softmax long-only split test_year=2017: train 2007-2016


c:\Users\ianning\AppData\Local\drl_env\Lib\site-packages\stable_baselines3\common\on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


softmax long-only split test_year=2018: train 2008-2017


c:\Users\ianning\AppData\Local\drl_env\Lib\site-packages\stable_baselines3\common\on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


softmax long-only split test_year=2019: train 2009-2018


...

softmax long-only split test_year=2025: train 2015-2024


c:\Users\ianning\AppData\Local\drl_env\Lib\site-packages\stable_baselines3\common\on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


softmax long-only split test_year=2026: train 2016-2025


c:\Users\ianning\AppData\Local\drl_env\Lib\site-packages\stable_baselines3\common\on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


,method,months,compound_return,sharpe_ratio,max_drawdown,sterling_ratio
0,CAFPO_PPO_SoftmaxLongOnly_T0.2_5SeedAvgAction,112,0.658971,0.114976,-0.293503,0.019462
1,EqualWeight_LongOnly,112,0.718658,0.121615,-0.282215,0.021334
2,Markowitz_LS,112,0.862838,0.202341,-0.232515,0.025825
3,ValueWeight_LongOnly,112,0.970738,0.162436,-0.258365,0.026980


## 6. Optional Structural Smoke Check

This checks the softmax mapping and environment mechanics without training PPO.

In [8]:
RUN_STRUCTURAL_SMOKE = True

if RUN_STRUCTURAL_SMOKE:
    s = splits[0]
    dummy_factors = np.zeros((len(arrays.dates), N_FACTORS), dtype=np.float32)
    env = make_longonly_env(dummy_factors, s.test_idx[:1])
    obs, _ = env.reset()
    action = np.linspace(-1.0, 1.0, arrays.x.shape[1], dtype=np.float32)
    obs2, reward, terminated, truncated, info = env.step(action)
    weights = info["weights"]
    print("obs shape:", obs.shape)
    print("weights min/max/sum:", float(weights.min()), float(weights.max()), float(weights.sum()))
    print("gross long/short:", info["gross_long"], info["gross_short"])
    print("effective_n:", info["effective_n"])
    assert obs.shape == (LOOKBACK, N_FACTORS)
    assert weights.min() >= -1e-8
    assert abs(weights.sum() - 1.0) < 1e-5
    assert info["gross_short"] == 0.0
else:
    print("RUN_STRUCTURAL_SMOKE is False.")

obs shape: (12, 5)
weights min/max/sum: 2.225126081611961e-06 0.04901166260242462 1.0
gross long/short: 1.0 0.0
effective_n: 39.804935455322266
